## O-NET Wrangling Notebook

Notebook for loading ONET data, organising, and saving to disk.

In [ ]:
#|hide
import nblite; from nbdev.showdoc import show_doc; nblite.nbl_export()

In [ ]:
import aisi_economy_index as proj
from aisi_economy_index import const

In [ ]:
import json
import asyncio
from typing import Optional, Tuple, Dict, Any, List
import adulib.llm as llm
import inspect

In [ ]:
from dotenv import load_dotenv

load_dotenv() 

True

In [ ]:
from pydantic import BaseModel, ValidationError, conint, confloat


## Loading ONET

In [ ]:
# Simple, direct approach - no helpers, no abstractions
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/bhargav/adu_dev/aisi-economy-index/aisi_economy_index/store/data/db_30_0_excel")
OUT_DIR = Path(str(const.data_path)) / "eval_dfs"
OUT_DIR.mkdir(exist_ok=True)

# Load reference tables
occupation_data = pd.read_excel(BASE_DIR / "Occupation Data.xlsx")
content_model_ref = pd.read_excel(BASE_DIR / "Content Model Reference.xlsx")

# Element ID → Name lookup
element_names = content_model_ref[["Element ID", "Element Name"]].drop_duplicates()
element_names.columns = ["ElementID", "ElementName"]

# SOC → Title lookup
titles = occupation_data[["O*NET-SOC Code", "Title"]].drop_duplicates()
titles.columns = ["OnetSocCode", "Title"]


def load_and_pivot(filename, out_name):
    """Load an O*NET file, pivot IM/LV to columns, join names."""
    df = pd.read_excel(BASE_DIR / filename)
    
    # Standardize column names (just the ones we need)
    df = df.rename(columns={
        "O*NET-SOC Code": "OnetSocCode",
        "Element ID": "ElementID",
        "Scale ID": "ScaleID",
        "Data Value": "DataValue",
    })
    
    # Pivot: one row per (SOC, Element), columns for importance/level
    pivoted = df.pivot_table(
        index=["OnetSocCode", "ElementID"],
        columns="ScaleID",
        values="DataValue",
        aggfunc="first"
    ).reset_index()
    
    # Rename IM/LV
    pivoted = pivoted.rename(columns={"IM": "importance", "LV": "level"})
    
    # Add element name
    pivoted = pivoted.merge(element_names, on="ElementID", how="left")
    
    # Add occupation title
    pivoted = pivoted.merge(titles, on="OnetSocCode", how="left")
    
    # Reorder and rename
    result = pivoted[["OnetSocCode", "Title", "ElementID", "ElementName", "importance", "level"]].copy()
    result = result.rename(columns={"ElementName": out_name})
    
    return result


# Build the four simple ones
skills_eval = load_and_pivot("Skills.xlsx", "skill")
abilities_eval = load_and_pivot("Abilities.xlsx", "ability")
knowledge_eval = load_and_pivot("Knowledge.xlsx", "knowledge_area")
gwas_eval = load_and_pivot("Work Activities.xlsx", "activity")

print(f"skills_eval: {len(skills_eval)} rows")
print(f"abilities_eval: {len(abilities_eval)} rows")
print(f"knowledge_eval: {len(knowledge_eval)} rows")
print(f"gwas_eval: {len(gwas_eval)} rows")

print("\nSample:")
print(skills_eval.head())

# Save
skills_eval.to_csv(OUT_DIR / "skills_eval.csv", index=False)
abilities_eval.to_csv(OUT_DIR / "abilities_eval.csv", index=False)
knowledge_eval.to_csv(OUT_DIR / "knowledge_eval.csv", index=False)
gwas_eval.to_csv(OUT_DIR / "gwas_eval.csv", index=False)

print(f"\nSaved to {OUT_DIR}")

skills_eval: 31290 rows
abilities_eval: 46488 rows
knowledge_eval: 29502 rows
gwas_eval: 36654 rows

Sample:
  OnetSocCode             Title ElementID                  skill  importance  \
0  11-1011.00  Chief Executives   2.A.1.a  Reading Comprehension        4.12   
1  11-1011.00  Chief Executives   2.A.1.b       Active Listening        4.00   
2  11-1011.00  Chief Executives   2.A.1.c                Writing        4.12   
3  11-1011.00  Chief Executives   2.A.1.d               Speaking        4.25   
4  11-1011.00  Chief Executives   2.A.1.e            Mathematics        3.25   

   level  
0   4.62  
1   4.75  
2   4.38  
3   4.75  
4   3.50  

Saved to /Users/bhargav/adu_dev/aisi-economy-index/aisi_economy_index/store/data/eval_dfs


In [ ]:
# ============================================================
# Work Context
# ============================================================

work_context_raw = pd.read_excel(BASE_DIR / "Work Context.xlsx")

work_context_raw = work_context_raw.rename(columns={
    "O*NET-SOC Code": "OnetSocCode",
    "Element ID": "ElementID",
    "Element Name": "ElementName",
    "Scale ID": "ScaleID",
    "Data Value": "DataValue",
})

# Filter to CX scale (1-5 continuous)
work_context_eval = work_context_raw[work_context_raw["ScaleID"] == "CX"].copy()

# Select columns (Title already exists in raw data)
work_context_eval = work_context_eval[[
    "OnetSocCode", "Title", "ElementID", "ElementName", "DataValue"
]].rename(columns={"DataValue": "context_value", "ElementName": "context_element"})

print(f"work_context_eval: {len(work_context_eval)} rows")
print(f"  Unique elements: {work_context_eval['ElementID'].nunique()}")
print(work_context_eval.head())

work_context_eval.to_csv(OUT_DIR / "work_context_eval.csv", index=False)

work_context_eval: 49170 rows
  Unique elements: 55
   OnetSocCode             Title    ElementID  \
0   11-1011.00  Chief Executives  4.C.1.a.2.c   
6   11-1011.00  Chief Executives  4.C.1.a.2.f   
12  11-1011.00  Chief Executives  4.C.1.a.2.h   
18  11-1011.00  Chief Executives  4.C.1.a.2.j   
24  11-1011.00  Chief Executives  4.C.1.a.2.l   

                                      context_element  context_value  
0                                     Public Speaking           3.07  
6                             Telephone Conversations           4.92  
12                                             E-Mail           4.97  
18                          Written Letters and Memos           4.14  
24  Face-to-Face Discussions with Individuals and ...           4.90  


In [ ]:
skills_eval

,OnetSocCode,Title,ElementID,skill,importance,level
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,4.12,4.62
1,11-1011.00,Chief Executives,2.A.1.b,Active Listening,4.00,4.75
2,11-1011.00,Chief Executives,2.A.1.c,Writing,4.12,4.38
3,11-1011.00,Chief Executives,2.A.1.d,Speaking,4.25,4.75
4,11-1011.00,Chief Executives,2.A.1.e,Mathematics,3.25,3.50
...,...,...,...,...,...,...
31285,53-7121.00,"Tank Car, Truck, and Ship Loaders",2.B.4.h,Systems Evaluation,2.12,2.12
31286,53-7121.00,"Tank Car, Truck, and Ship Loaders",2.B.5.a,Time Management,3.12,2.88
31287,53-7121.00,"Tank Car, Truck, and Ship Loaders",2.B.5.b,Management of Financial Resources,2.00,1.12
31288,53-7121.00,"Tank Car, Truck, and Ship Loaders",2.B.5.c,Management of Material Resources,2.00,1.88


In [ ]:
# ============================================================
# DWAs (Detailed Work Activities)
# ============================================================

tasks_to_dwas = pd.read_excel(BASE_DIR / "Tasks to DWAs.xlsx")
task_ratings = pd.read_excel(BASE_DIR / "Task Ratings.xlsx")

# Standardize columns
tasks_to_dwas = tasks_to_dwas.rename(columns={
    "O*NET-SOC Code": "OnetSocCode",
    "Task ID": "TaskID",
    "DWA ID": "DWA_ID",
    "DWA Title": "DWA_Title",
})

task_ratings = task_ratings.rename(columns={
    "O*NET-SOC Code": "OnetSocCode",
    "Task ID": "TaskID",
    "Scale ID": "ScaleID",
    "Data Value": "DataValue",
})

# Get task importance (for weighting)
task_importance = task_ratings[task_ratings["ScaleID"] == "IM"][["OnetSocCode", "TaskID", "DataValue"]]
task_importance = task_importance.rename(columns={"DataValue": "task_weight"})

# Join weight to tasks_to_dwas
dwas_with_weights = tasks_to_dwas.merge(task_importance, on=["OnetSocCode", "TaskID"], how="left")
dwas_with_weights["task_weight"] = dwas_with_weights["task_weight"].fillna(1.0)

# Aggregate: sum weights per (SOC, DWA)
dwas_eval = (
    dwas_with_weights
    .groupby(["OnetSocCode", "DWA_ID", "DWA_Title"], as_index=False)["task_weight"]
    .sum()
    .rename(columns={"task_weight": "weight"})
)

# Normalize within occupation
totals = dwas_eval.groupby("OnetSocCode")["weight"].transform("sum")
dwas_eval["weight_norm"] = dwas_eval["weight"] / totals

# Add title
dwas_eval = dwas_eval.merge(titles, on="OnetSocCode", how="left")

# Reorder
dwas_eval = dwas_eval[["OnetSocCode", "Title", "DWA_ID", "DWA_Title", "weight", "weight_norm"]]

print(f"dwas_eval: {len(dwas_eval)} rows")
print(dwas_eval.head())

dwas_eval.to_csv(OUT_DIR / "dwas_eval.csv", index=False)


# ============================================================
# Tasks
# ============================================================

task_statements = pd.read_excel(BASE_DIR / "Task Statements.xlsx")

task_statements = task_statements.rename(columns={
    "O*NET-SOC Code": "OnetSocCode",
    "Task ID": "TaskID",
    "Task": "Task",
    "Task Type": "Task_Type",
})

# Pivot task ratings to get Importance, Relevance
# (Frequency is more complex - it's a distribution, so we'll just get the mean DataValue)

# Importance
importance = task_ratings[task_ratings["ScaleID"] == "IM"][["OnetSocCode", "TaskID", "DataValue"]]
importance = importance.groupby(["OnetSocCode", "TaskID"], as_index=False).mean()
importance = importance.rename(columns={"DataValue": "Task_Importance"})

# Relevance
relevance = task_ratings[task_ratings["ScaleID"] == "RT"][["OnetSocCode", "TaskID", "DataValue"]]
relevance = relevance.groupby(["OnetSocCode", "TaskID"], as_index=False).mean()
relevance = relevance.rename(columns={"DataValue": "Task_Relevance"})

# Frequency (FT scale) - weighted mean using Category as the value
freq_data = task_ratings[task_ratings["ScaleID"] == "FT"].copy()
if not freq_data.empty and "Category" in freq_data.columns:
    freq_data["Category"] = pd.to_numeric(freq_data["Category"], errors="coerce")
    freq_data["weighted"] = freq_data["Category"] * freq_data["DataValue"] / 100  # DataValue is percentage
    frequency = freq_data.groupby(["OnetSocCode", "TaskID"], as_index=False)["weighted"].sum()
    frequency = frequency.rename(columns={"weighted": "Task_FrequencyMean"})
else:
    frequency = pd.DataFrame(columns=["OnetSocCode", "TaskID", "Task_FrequencyMean"])

# Build tasks_eval
tasks_eval = task_statements[["OnetSocCode", "TaskID", "Task", "Task_Type"]].copy()

# Add ratings
tasks_eval = tasks_eval.merge(importance, on=["OnetSocCode", "TaskID"], how="left")
tasks_eval = tasks_eval.merge(relevance, on=["OnetSocCode", "TaskID"], how="left")
tasks_eval = tasks_eval.merge(frequency, on=["OnetSocCode", "TaskID"], how="left")

# Add DWA link (take first DWA if multiple)
dwa_link = tasks_to_dwas[["OnetSocCode", "TaskID", "DWA_ID", "DWA_Title"]].drop_duplicates()
tasks_eval = tasks_eval.merge(dwa_link, on=["OnetSocCode", "TaskID"], how="left")

# Add title
tasks_eval = tasks_eval.merge(titles, on="OnetSocCode", how="left")

# Reorder
tasks_eval = tasks_eval[["OnetSocCode", "Title", "TaskID", "Task", "Task_Type", 
                          "DWA_ID", "DWA_Title", "Task_Importance", "Task_FrequencyMean", "Task_Relevance"]]

print(f"\ntasks_eval: {len(tasks_eval)} rows")
print(tasks_eval.head())

tasks_eval.to_csv(OUT_DIR / "tasks_eval.csv", index=False)


# ============================================================
# Technology Skills (software)
# ============================================================

tech_skills = pd.read_excel(BASE_DIR / "Technology Skills.xlsx")

tech_skills = tech_skills.rename(columns={
    "O*NET-SOC Code": "OnetSocCode",
    "Commodity Code": "CommodityCode",
    "Commodity Title": "CommodityTitle",
    "Hot Technology": "HotTechnology",
    "In Demand": "InDemand",
})

# Add title (if not already there with correct name)
if "Title" not in tech_skills.columns:
    tech_skills = tech_skills.merge(titles, on="OnetSocCode", how="left")

# Clean Y/N columns
for col in ["HotTechnology", "InDemand"]:
    if col in tech_skills.columns:
        tech_skills[col] = tech_skills[col].map({"Y": 1, "N": 0}).fillna(0).astype(int)

tech_skills_eval = tech_skills[["OnetSocCode", "Title", "CommodityCode", "CommodityTitle", "Example", "HotTechnology", "InDemand"]]

print(f"tech_skills_eval: {len(tech_skills_eval)} rows")
print(tech_skills_eval.head())

tech_skills_eval.to_csv(OUT_DIR / "tech_skills_eval.csv", index=False)


# ============================================================
# Tools Used (physical tools)
# ============================================================

tools_used = pd.read_excel(BASE_DIR / "Tools Used.xlsx")

tools_used = tools_used.rename(columns={
    "O*NET-SOC Code": "OnetSocCode",
    "Commodity Code": "CommodityCode",
    "Commodity Title": "CommodityTitle",
})

# Add title if needed
if "Title" not in tools_used.columns:
    tools_used = tools_used.merge(titles, on="OnetSocCode", how="left")

tools_eval = tools_used[["OnetSocCode", "Title", "CommodityCode", "CommodityTitle", "Example"]]

print(f"\ntools_eval: {len(tools_eval)} rows")
print(tools_eval.head())

tools_eval.to_csv(OUT_DIR / "tools_eval.csv", index=False)



# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"skills_eval:      {len(skills_eval):>6} rows")
print(f"abilities_eval:   {len(abilities_eval):>6} rows")
print(f"knowledge_eval:   {len(knowledge_eval):>6} rows")
print(f"gwas_eval:        {len(gwas_eval):>6} rows")
print(f"dwas_eval:        {len(dwas_eval):>6} rows")
print(f"tasks_eval:       {len(tasks_eval):>6} rows")
print(f"tech_skills_eval: {len(tech_skills_eval):>6} rows")
print(f"tools_eval:       {len(tools_eval):>6} rows")
print("=" * 60)
print(f"All saved to: {OUT_DIR}")

dwas_eval: 18358 rows
  OnetSocCode             Title             DWA_ID  \
0  11-1011.00  Chief Executives  4.A.1.a.1.I03.D06   
1  11-1011.00  Chief Executives  4.A.1.a.1.I18.D03   
2  11-1011.00  Chief Executives  4.A.1.a.1.I20.D04   
3  11-1011.00  Chief Executives  4.A.2.a.4.I07.D09   
4  11-1011.00  Chief Executives  4.A.2.a.4.I07.D12   

                                           DWA_Title  weight  weight_norm  
0      Conduct hearings to investigate legal issues.    3.98     0.023979  
1                 Conduct research on social issues.    3.66     0.022051  
2  Conduct research to gain information about pro...    3.66     0.022051  
3  Analyze data to assess operational or project ...    4.13     0.024883  
4  Analyze data to inform operational decisions o...    4.07     0.024521  

tasks_eval: 23851 rows
  OnetSocCode             Title  TaskID  \
0  11-1011.00  Chief Executives    8823   
1  11-1011.00  Chief Executives    8824   
2  11-1011.00  Chief Executives    8827   
3

In [ ]:
# ============================================================
# Descriptor Definitions (for Felten-style mapping)
# ============================================================

# Check what columns are in content_model_ref
print("Content Model Reference columns:")
print(content_model_ref.columns.tolist())
print(f"\nShape: {content_model_ref.shape}")
print(content_model_ref.head())

Content Model Reference columns:
['Element ID', 'Element Name', 'Description']

Shape: (627, 3)
  Element ID            Element Name  \
0          1  Worker Characteristics   
1        1.A               Abilities   
2      1.A.1     Cognitive Abilities   
3    1.A.1.a        Verbal Abilities   
4  1.A.1.a.1      Oral Comprehension   

                                         Description  
0                             Worker Characteristics  
1  Enduring attributes of the individual that inf...  
2  Abilities that influence the acquisition and a...  
3  Abilities that influence the acquisition and a...  
4  The ability to listen to and understand inform...  


In [ ]:
# Extract definitions
# The file typically has: Element ID, Element Name, Description

descriptor_definitions = content_model_ref[["Element ID", "Element Name", "Description"]].copy()
descriptor_definitions.columns = ["ElementID", "ElementName", "Definition"]
descriptor_definitions = descriptor_definitions.drop_duplicates()

# Filter to just Abilities, Skills, Knowledge based on ElementID prefix
# Abilities: 1.A.*
# Skills: 2.A.* and 2.B.*
# Knowledge: 2.C.*

def get_descriptor_block(element_id: str) -> str:
    """Classify ElementID into block type."""
    if element_id.startswith("1.A."):
        return "ability"
    elif element_id.startswith("2.A.") or element_id.startswith("2.B."):
        return "skill"
    elif element_id.startswith("2.C."):
        return "knowledge"
    elif element_id.startswith("4.A."):
        return "work_activity"  # GWAs, in case you want them later
    else:
        return "other"

descriptor_definitions["Block"] = descriptor_definitions["ElementID"].apply(get_descriptor_block)

# Check counts
print("\nDescriptor counts by block:")
print(descriptor_definitions["Block"].value_counts())

# Filter to just the three we need
descriptor_definitions_filtered = descriptor_definitions[
    descriptor_definitions["Block"].isin(["ability", "skill", "knowledge"])
].copy()

print(f"\nFiltered to {len(descriptor_definitions_filtered)} descriptors")
print(descriptor_definitions_filtered.head(10))


Descriptor counts by block:
Block
other            419
ability           71
work_activity     54
skill             42
knowledge         41
Name: count, dtype: int64

Filtered to 154 descriptors
    ElementID                              ElementName  \
2       1.A.1                      Cognitive Abilities   
3     1.A.1.a                         Verbal Abilities   
4   1.A.1.a.1                       Oral Comprehension   
5   1.A.1.a.2                    Written Comprehension   
6   1.A.1.a.3                          Oral Expression   
7   1.A.1.a.4                       Written Expression   
8     1.A.1.b  Idea Generation and Reasoning Abilities   
9   1.A.1.b.1                         Fluency of Ideas   
10  1.A.1.b.2                              Originality   
11  1.A.1.b.3                      Problem Sensitivity   

                                           Definition    Block  
2   Abilities that influence the acquisition and a...  ability  
3   Abilities that influence the acq

In [ ]:
# Check for missing definitions
missing = descriptor_definitions_filtered[descriptor_definitions_filtered["Definition"].isna()]
print(f"\nMissing definitions: {len(missing)}")
if len(missing) > 0:
    print(missing[["ElementID", "ElementName"]].head(10))


Missing definitions: 0


In [ ]:
# Save to disk
descriptor_definitions_filtered.to_csv(OUT_DIR / "descriptor_definitions.csv", index=False)

# Also save as a simple dict for easy loading (ElementID -> Definition)
definitions_dict = dict(zip(
    descriptor_definitions_filtered["ElementID"], 
    descriptor_definitions_filtered["Definition"].fillna("")
))

import json
with open(OUT_DIR / "descriptor_definitions.json", "w") as f:
    json.dump(definitions_dict, f, indent=2)

print(f"\nSaved to:")
print(f"  {OUT_DIR / 'descriptor_definitions.csv'}")
print(f"  {OUT_DIR / 'descriptor_definitions.json'}")


Saved to:
  /Users/bhargav/adu_dev/aisi-economy-index/aisi_economy_index/store/data/eval_dfs/descriptor_definitions.csv
  /Users/bhargav/adu_dev/aisi-economy-index/aisi_economy_index/store/data/eval_dfs/descriptor_definitions.json


In [ ]:
# Quick sanity check - show a few examples
print("\n" + "=" * 60)
print("SAMPLE DEFINITIONS")
print("=" * 60)

for block in ["ability", "skill", "knowledge"]:
    sample = descriptor_definitions_filtered[descriptor_definitions_filtered["Block"] == block].head(2)
    print(f"\n{block.upper()}:")
    for _, row in sample.iterrows():
        print(f"  {row['ElementName']}:")
        print(f"    {row['Definition'][:100]}..." if len(str(row['Definition'])) > 100 else f"    {row['Definition']}")


SAMPLE DEFINITIONS

ABILITY:
  Cognitive Abilities:
    Abilities that influence the acquisition and application of knowledge in problem solving
  Verbal Abilities:
    Abilities that influence the acquisition and application of verbal information in problem solving

SKILL:
  Content:
    Background structures needed to work with and acquire more specific skills in a variety of different...
  Reading Comprehension:
    Understanding written sentences and paragraphs in work-related documents.

KNOWLEDGE:
  Business and Management:
    Knowledge of principles and facts related to business administration and accounting, human and mater...
  Administration and Management:
    Knowledge of business and management principles involved in strategic planning, resource allocation,...


## Exploring O-NET by SOC code

Helper functions to explore O-NET data for specific SOC codes.

In [ ]:
import os
import pandas as pd

EVAL_DF_DIR = const.data_path / "eval_dfs"

def _load_csv(path):
    if not os.path.exists(path):
        # Return empty DF with no error if a file is missing
        return pd.DataFrame()
    return pd.read_csv(path)

# Load the new eval tables
skills_eval     = _load_csv(f"{EVAL_DF_DIR}/skills_eval.csv")
abilities_eval  = _load_csv(f"{EVAL_DF_DIR}/abilities_eval.csv")
knowledge_eval  = _load_csv(f"{EVAL_DF_DIR}/knowledge_eval.csv")
gwas_eval       = _load_csv(f"{EVAL_DF_DIR}/gwas_eval.csv")      # was activities_eval
dwas_eval       = _load_csv(f"{EVAL_DF_DIR}/dwas_eval.csv")      # new
tasks_eval      = _load_csv(f"{EVAL_DF_DIR}/tasks_eval.csv")

# Bundle for pipelines (with a compat alias for activities_eval → gwas_eval)
eval_dfs = dict(
    skills_eval=skills_eval,
    abilities_eval=abilities_eval,
    knowledge_eval=knowledge_eval,
    gwas_eval=gwas_eval,
    dwas_eval=dwas_eval,
    tasks_eval=tasks_eval,

    # Backwards-compat key (some old code may still look for this)
    activities_eval=gwas_eval,
)


In [ ]:
def summarize_soc_counts(soc_code: str, eval_dfs: dict, verbose: bool = True):
    """
    Summarize how many skills, abilities, knowledge areas, GWAs (activities),
    DWAs, and tasks exist for a SOC in your eval_dfs.
    """
    summary = {"soc": soc_code, "title": None}

    def _first_nonempty_df(*candidates):
        for key in candidates:
            df = eval_dfs.get(key)
            if isinstance(df, pd.DataFrame) and not df.empty:
                return df
        return pd.DataFrame()

    def _count_unique(df, soc_col, target_cols):
        if df is None or df.empty or soc_col not in df.columns:
            return 0, None
        sub = df[df[soc_col] == soc_code]
        if sub.empty:
            return 0, sub
        for c in target_cols:
            if c in sub.columns:
                return sub[c].nunique(dropna=True), sub
        return 0, sub

    def _maybe_set_title(df):
        if summary["title"] is None and isinstance(df, pd.DataFrame) and not df.empty and "Title" in df.columns:
            t = df["Title"].dropna()
            if not t.empty:
                summary["title"] = t.iloc[0]

    # Skills
    df = _first_nonempty_df("skills_eval")
    c, sub = _count_unique(df, "OnetSocCode", ["skill", "Skill", "Element Name"])
    summary["skills"] = c; _maybe_set_title(sub)

    # Abilities
    df = _first_nonempty_df("abilities_eval")
    c, sub = _count_unique(df, "OnetSocCode", ["ability", "Ability", "Element Name"])
    summary["abilities"] = c; _maybe_set_title(sub)

    # Knowledge
    df = _first_nonempty_df("knowledge_eval")
    c, sub = _count_unique(df, "OnetSocCode", ["knowledge_area", "Knowledge", "Element Name"])
    summary["knowledge"] = c; _maybe_set_title(sub)

    # Activities (GWAs) — prefer gwas_eval, fall back to activities_eval
    df = _first_nonempty_df("gwas_eval", "activities_eval")
    c, sub = _count_unique(df, "OnetSocCode", ["activity", "Activity", "Element Name"])
    summary["activities"] = c; _maybe_set_title(sub)

    # DWAs
    df = _first_nonempty_df("dwas_eval")
    c, sub = _count_unique(df, "OnetSocCode", ["DWA_ID", "DwaID", "DWA_Title", "DWA Title"])
    summary["dwas"] = c; _maybe_set_title(sub)

    # Tasks
    df = _first_nonempty_df("tasks_eval")
    c, sub = _count_unique(df, "OnetSocCode", ["TaskID", "Task"])
    summary["tasks"] = c; _maybe_set_title(sub)

    if verbose:
        print(f"Summary for {soc_code} ({summary.get('title') or 'Unknown title'})")
        print("───────────────────────────────────────────────")
        for k in ["skills", "abilities", "knowledge", "activities", "dwas", "tasks"]:
            print(f"{k.capitalize():<12}: {summary[k]}")
        print("───────────────────────────────────────────────")

    return summary


In [ ]:
# Example usage
summarize_soc_counts("11-1011.00", eval_dfs)

In [ ]:
summarize_soc_counts("53-7121.00", eval_dfs)

In [ ]:
# ---------- Helpers for printing SOC items

def _pick_df(eval_dfs: dict, *candidates):
    """Return first non-empty DataFrame among candidate keys (or empty DF)."""
    for key in candidates:
        df = eval_dfs.get(key)
        if isinstance(df, pd.DataFrame) and not df.empty:
            return df
    return pd.DataFrame()

def _first_col(df: pd.DataFrame, candidates):
    """Return the first column from candidates that exists in df (or None)."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _subset_by_soc(df: pd.DataFrame, soc_code: str):
    """Subset DF to a SOC, trying common SOC column names."""
    soc_col = _first_col(df, ["OnetSocCode", "O*NET-SOC Code", "O*NET-SOC Code "])
    if not soc_col or df.empty:
        return pd.DataFrame()
    return df[df[soc_col] == soc_code]

def _maybe_title(df: pd.DataFrame):
    """Try to extract an occupation Title from a DF if present."""
    if isinstance(df, pd.DataFrame) and not df.empty:
        title_col = _first_col(df, ["Title", "Occupation", "Occupation Title"])
        if title_col:
            t = df[title_col].dropna()
            if not t.empty:
                return str(t.iloc[0])
    return None

def _get_items_with_scores(df: pd.DataFrame, name_candidates: list, score_cols: list = None):
    """
    Return list of dicts: [{name: ..., importance: ..., level: ...}, ...]
    If score_cols not found, just returns names as strings (backward compat).
    """
    if df is None or df.empty:
        return []
    
    name_col = _first_col(df, name_candidates)
    if not name_col:
        return []
    
    # Check if we have score columns
    available_scores = [c for c in (score_cols or []) if c in df.columns]
    
    if not available_scores:
        # Fallback: just return names (original behavior)
        return sorted(df[name_col].dropna().astype(str).unique().tolist())
    
    # Return list of dicts with name + scores
    cols = [name_col] + available_scores
    sub = df[cols].drop_duplicates().dropna(subset=[name_col])
    sub = sub.sort_values(name_col)
    
    result = []
    for _, row in sub.iterrows():
        item = {"name": row[name_col]}
        for sc in available_scores:
            item[sc] = row[sc] if pd.notna(row[sc]) else None
        result.append(item)
    return result


def print_soc_items(soc_code: str, eval_dfs: dict, show_scores: bool = True):
    """
    Print all items (and counts) associated with a SOC.
    If show_scores=True, also prints importance/level/weight.
    """
    title = None

    # Skills
    df_skills = _pick_df(eval_dfs, "skills_eval")
    sub_skills = _subset_by_soc(df_skills, soc_code)
    title = title or _maybe_title(sub_skills)
    skills = _get_items_with_scores(
        sub_skills,
        ["skill", "Skill", "Element Name"],
        ["importance", "level"] if show_scores else None
    )

    # Abilities
    df_abilities = _pick_df(eval_dfs, "abilities_eval")
    sub_abilities = _subset_by_soc(df_abilities, soc_code)
    title = title or _maybe_title(sub_abilities)
    abilities = _get_items_with_scores(
        sub_abilities,
        ["ability", "Ability", "Element Name"],
        ["importance", "level"] if show_scores else None
    )

    # Knowledge
    df_knowledge = _pick_df(eval_dfs, "knowledge_eval")
    sub_knowledge = _subset_by_soc(df_knowledge, soc_code)
    title = title or _maybe_title(sub_knowledge)
    knowledge = _get_items_with_scores(
        sub_knowledge,
        ["knowledge_area", "Knowledge", "Element Name"],
        ["importance", "level"] if show_scores else None
    )

    # Activities (GWAs)
    df_gwas = _pick_df(eval_dfs, "gwas_eval", "activities_eval")
    sub_gwas = _subset_by_soc(df_gwas, soc_code)
    title = title or _maybe_title(sub_gwas)
    activities = _get_items_with_scores(
        sub_gwas,
        ["activity", "Activity", "Element Name"],
        ["importance", "level"] if show_scores else None
    )

    # DWAs
    df_dwas = _pick_df(eval_dfs, "dwas_eval")
    sub_dwas = _subset_by_soc(df_dwas, soc_code)
    title = title or _maybe_title(sub_dwas)
    dwas = _get_items_with_scores(
        sub_dwas,
        ["DWA_Title", "DWA Title"],
        ["weight", "weight_norm"] if show_scores else None
    )
    if not dwas:  # fallback to IDs
        dwas = _get_items_with_scores(sub_dwas, ["DWA_ID", "DwaID"], None)

    # Tasks
    df_tasks = _pick_df(eval_dfs, "tasks_eval")
    sub_tasks = _subset_by_soc(df_tasks, soc_code)
    title = title or _maybe_title(sub_tasks)
    tasks = _get_items_with_scores(
        sub_tasks,
        ["Task", "Task Description"],
        ["Task_Importance", "Task_Relevance"] if show_scores else None
    )
    if not tasks:
        tasks = _get_items_with_scores(sub_tasks, ["TaskID"], None)

    # ---------- Print helper
    def _print_section(name: str, items: list, score_labels: list = None):
        print(f"\n{name:<12}: {len(items)}")
        for i, x in enumerate(items, 1):
            if isinstance(x, dict):
                scores_str = ""
                if score_labels and show_scores:
                    parts = []
                    for sl in score_labels:
                        val = x.get(sl)
                        if val is not None:
                            parts.append(f"{sl}={val:.2f}" if isinstance(val, float) else f"{sl}={val}")
                    if parts:
                        scores_str = f"  ({', '.join(parts)})"
                print(f"  {i:>3}. {x['name']}{scores_str}")
            else:
                print(f"  {i:>3}. {x}")

    # ---------- Print
    header_title = title or "Unknown title"
    print(f"Items for {soc_code} ({header_title})")
    print("───────────────────────────────────────────────")
    
    _print_section("Skills", skills, ["importance", "level"])
    _print_section("Abilities", abilities, ["importance", "level"])
    _print_section("Knowledge", knowledge, ["importance", "level"])
    _print_section("Activities", activities, ["importance", "level"])
    _print_section("DWAs", dwas, ["weight", "weight_norm"])
    _print_section("Tasks", tasks, ["Task_Importance", "Task_Relevance"])
    
    print("───────────────────────────────────────────────")


In [ ]:
# Example: print all items for Chief Executives
print_soc_items("11-1011.00", eval_dfs)